# Anti-UAV Detection Pipeline — Cloud GPU Launcher (Colab / Kaggle)

**Title:** *Small Targets at Speed: Input Resolution and RPN Anchor Scale Sensitivity in Vision-Based Anti-UAV Detection*  
**Repository:** `https://github.com/Lc-wing/CV_final_project.git`

This thin launcher notebook auto-detects **Google Colab** (extracting the dataset onto fast local SSD at `/content/data/dut_anti_uav` to avoid Google Drive FUSE small-file bottlenecks while mounting `/content/drive/MyDrive/CV_project` so checkpoints, raw CSV logs, metrics, figures, and test-pack predictions survive disconnects) or **Kaggle Notebooks** (`/kaggle/working/data/dut_anti_uav` and `/kaggle/working/CV_project`), then invokes the exact CLI commands documented in `README.md` with live streamed output for all three controlled runs:

1. `res_800` *(Baseline)*: `min_size=800, max_size=1333`, anchors `(32, 64, 128, 256, 512)`
2. `res_480` *(Variant 1)*: `min_size=480, max_size=800`, anchors `(32, 64, 128, 256, 512)`
3. `res_480_small_anchor` *(Variant 2)*: `min_size=480, max_size=800`, anchors `(16, 32, 64, 128, 256)`

## 1. Environment Auto-Detection, Streaming Subprocess Helper & Storage Setup

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

def run_cmd(cmd):
    """Run a subprocess command, streaming stdout/stderr live line-by-line."""
    cmd_list = [str(c) for c in cmd]
    print("Running:", " ".join(cmd_list), flush=True)
    with subprocess.Popen(
        cmd_list,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    ) as proc:
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="", flush=True)
        rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd_list)

IN_COLAB = False
IN_KAGGLE = Path("/kaggle/working").exists()

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive", force_remount=False)
    STORAGE_ROOT = Path("/content/drive/MyDrive/CV_project")
    # Fast local SSD avoids Drive FUSE small-file bottlenecks across ~20k JPEG/XML files.
    # Optional raw split zips can be mirrored separately under STORAGE_ROOT / "data".
    DATA_DIR = Path("/content/data/dut_anti_uav")
    REPO_DIR = Path("/content/CV_final_project")
    PLATFORM = "Google Colab"
elif IN_KAGGLE:
    STORAGE_ROOT = Path("/kaggle/working/CV_project")
    DATA_DIR = Path("/kaggle/working/data/dut_anti_uav")
    REPO_DIR = Path("/kaggle/working/CV_final_project")
    PLATFORM = "Kaggle"
else:
    STORAGE_ROOT = Path.cwd()
    DATA_DIR = Path("data/dut_anti_uav")
    REPO_DIR = Path.cwd()
    PLATFORM = "Local"

CKPT_DIR = STORAGE_ROOT / "checkpoints"
RESULTS_DIR = STORAGE_ROOT / "results"
LOG_DIR = RESULTS_DIR / "logs"
METRICS_DIR = RESULTS_DIR / "metrics"
FIGURES_DIR = RESULTS_DIR / "figures"
PREDICTIONS_DIR = STORAGE_ROOT / "submission_pack" / "predictions"

for d in (DATA_DIR, CKPT_DIR, LOG_DIR, METRICS_DIR, FIGURES_DIR, PREDICTIONS_DIR):
    d.mkdir(parents=True, exist_ok=True)

os.environ["STORAGE_ROOT"] = str(STORAGE_ROOT)
os.environ["DATA_DIR"] = str(DATA_DIR)
os.environ["CKPT_DIR"] = str(CKPT_DIR)
os.environ["RESULTS_DIR"] = str(RESULTS_DIR)
os.environ["LOG_DIR"] = str(LOG_DIR)
os.environ["METRICS_DIR"] = str(METRICS_DIR)
os.environ["FIGURES_DIR"] = str(FIGURES_DIR)
os.environ["PREDICTIONS_DIR"] = str(PREDICTIONS_DIR)

print(f"Platform:        {PLATFORM}")
print(f"REPO_DIR:        {REPO_DIR}")
print(f"STORAGE_ROOT:    {STORAGE_ROOT}")
print(f"DATA_DIR:        {DATA_DIR}")
print(f"CKPT_DIR:        {CKPT_DIR}")
print(f"RESULTS_DIR:     {RESULTS_DIR}")
print(f"PREDICTIONS_DIR: {PREDICTIONS_DIR}")

try:
    run_cmd(["nvidia-smi"])
except (FileNotFoundError, subprocess.CalledProcessError):
    print("nvidia-smi not available (CPU runtime)")

## 2. Clone / Pull Repository & Install Dependencies

In [ ]:
import os
import sys

REPO_URL = "https://github.com/Lc-wing/CV_final_project.git"
BRANCH = "main"

if PLATFORM != "Local":
    if not (REPO_DIR / ".git").exists():
        run_cmd(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)])
    else:
        run_cmd(["git", "-C", str(REPO_DIR), "fetch", "origin"])
        run_cmd(["git", "-C", str(REPO_DIR), "checkout", BRANCH])
        run_cmd(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH])

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

run_cmd([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

import torch
import torchvision

print(f"Working dir:  {os.getcwd()}")
print(f"PyTorch:      {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device:   {torch.cuda.get_device_name(0)}")
print(f"TorchVision:  {torchvision.__version__}")

## 3. Download DUT Anti-UAV Dataset & Run `train + val` EDA

Idempotently downloads and extracts `aegean-ai/dut-anti-uav` (Apache-2.0) onto fast local storage (`$DATA_DIR`, verifying both `img/` and `xml/` before skipping) and computes original-coordinate scale bin edges strictly from `train + val` (never touching `test`), persisting `eda_train_val.json` and sibling `scale_bins.json` into `$METRICS_DIR`.

In [ ]:
import json
import sys

eda_out = METRICS_DIR / "eda_train_val.json"
cmd = [
    sys.executable, "-m", "src.prepare_data",
    "--data-dir", str(DATA_DIR),
    "--eda-out", str(eda_out),
]
run_cmd(cmd)

scale_bins_path = METRICS_DIR / "scale_bins.json"
print("scale_bins.json:")
print(json.dumps(json.loads(scale_bins_path.read_text(encoding="utf-8")), indent=2))

## 4. Fast Smoke Test & Timing Check (Run First)

Verifies CUDA AMP forward/backward/step execution, CSV logging, and checkpoint saving in under a minute before launching the 12-epoch training runs. Outputs are isolated under `$STORAGE_ROOT/smoke_runs/` so smoke CSVs never pollute `$RESULTS_DIR`.

In [ ]:
import sys
import time

smoke_logs = STORAGE_ROOT / "smoke_runs" / "logs"
smoke_ckpts = STORAGE_ROOT / "smoke_runs" / "checkpoints"

t0 = time.perf_counter()
run_cmd([
    sys.executable, "-m", "src.train",
    "--config", "configs/res_800.yaml",
    "--data-dir", str(DATA_DIR),
    "--log-dir", str(smoke_logs),
    "--checkpoint-dir", str(smoke_ckpts),
    "--smoke-test",
])
print(f"Smoke test completed cleanly in {time.perf_counter() - t0:.1f}s")

## 5. Train All 3 Controlled Runs (Automatic `--resume` on Disconnect)

Each run writes raw step/epoch CSV logs to `$LOG_DIR` and saves `<run>_last.pt` and `<run>_best.pt` (selected strictly by validation `AP50`) to `$CKPT_DIR` after every epoch. If Colab disconnects mid-run, re-running this cell automatically passes `--resume` whenever `<run>_last.pt` already exists.

In [ ]:
import sys

RUN_NAMES = ["res_800", "res_480", "res_480_small_anchor"]

for run_name in RUN_NAMES:
    cmd = [
        sys.executable, "-m", "src.train",
        "--config", f"configs/{run_name}.yaml",
        "--data-dir", str(DATA_DIR),
        "--log-dir", str(LOG_DIR),
        "--checkpoint-dir", str(CKPT_DIR),
    ]
    if (CKPT_DIR / f"{run_name}_last.pt").is_file():
        cmd.append("--resume")
        print(f"[{run_name}] Found existing _last.pt -> resuming", flush=True)
    else:
        print(f"[{run_name}] Starting fresh training run", flush=True)

    run_cmd(cmd)

## 6. Test-Split Scale-Stratified Evaluation & Latency Benchmark

In [ ]:
import sys

scale_bins_path = METRICS_DIR / "scale_bins.json"

for run_name in ["res_800", "res_480", "res_480_small_anchor"]:
    cmd = [
        sys.executable, "-m", "src.eval",
        "--config", f"configs/{run_name}.yaml",
        "--checkpoint", str(CKPT_DIR / f"{run_name}_best.pt"),
        "--data-dir", str(DATA_DIR),
        "--split", "test",
        "--scale-bins", str(scale_bins_path),
        "--out", str(METRICS_DIR / f"{run_name}_test.json"),
    ]
    run_cmd(cmd)

## 7. Regenerate Training Loss Curve & Scale-Recall / Latency Comparison Figures

In [ ]:
import sys

cmd = [
    sys.executable, "-m", "src.plot_curves",
    "--logs",
    str(LOG_DIR / "res_800_epoch_log.csv"),
    str(LOG_DIR / "res_480_epoch_log.csv"),
    str(LOG_DIR / "res_480_small_anchor_epoch_log.csv"),
    "--loss-out", str(FIGURES_DIR / "training_loss_curves.png"),
    "--x-axis", "epoch",
    "--eval-jsons",
    str(METRICS_DIR / "res_800_test.json"),
    str(METRICS_DIR / "res_480_test.json"),
    str(METRICS_DIR / "res_480_small_anchor_test.json"),
    "--compare-out", str(FIGURES_DIR / "scale_recall_and_latency.png"),
]
run_cmd(cmd)

## 8. Small Test-Pack Predictions, Comparison Visuals & Single-Image Inference

In [ ]:
import sys

IMAGE_EXTS = {".jpg", ".jpeg", ".png"}
storage_test_pack = STORAGE_ROOT / "submission_pack" / "test_pack"
repo_test_pack = REPO_DIR / "submission_pack" / "test_pack"

def _list_pack_images(pack_dir):
    img_dir = pack_dir / "img"
    if not img_dir.is_dir():
        return []
    return sorted(
        p for p in img_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    )

test_pack_dir = storage_test_pack if _list_pack_images(storage_test_pack) else repo_test_pack
pack_images = _list_pack_images(test_pack_dir)
out_csv = PREDICTIONS_DIR / "test_pack_predictions.csv"
visuals_dir = PREDICTIONS_DIR / "visuals"

if pack_images:
    pack_cmd = [
        sys.executable, "-m", "src.predict_pack",
        "--run", "configs/res_800.yaml", str(CKPT_DIR / "res_800_best.pt"),
        "--run", "configs/res_480.yaml", str(CKPT_DIR / "res_480_best.pt"),
        "--run", "configs/res_480_small_anchor.yaml", str(CKPT_DIR / "res_480_small_anchor_best.pt"),
        "--test-pack-dir", str(test_pack_dir),
        "--out-csv", str(out_csv),
        "--visuals-dir", str(visuals_dir),
    ]
    run_cmd(pack_cmd)

    sample_img = pack_images[0]
    single_cmd = [
        sys.executable, "-m", "src.predict_single",
        "--config", "configs/res_800.yaml",
        "--checkpoint", str(CKPT_DIR / "res_800_best.pt"),
        "--image", str(sample_img),
        "--score-thresh", "0.50",
        "--out-image", str(FIGURES_DIR / f"sample_pred_{sample_img.stem}.png"),
    ]
    run_cmd(single_cmd)
else:
    print(
        f"Populate {storage_test_pack}/{{img,xml}} or {repo_test_pack}/{{img,xml}} "
        "with curated test samples first."
    )

## 9. Print Checkpoint SHA-256 Checksums (for `checkpoints_manifest.json`)

In [ ]:
from scripts.download_checkpoints import compute_sha256

for run_name in ["res_800", "res_480", "res_480_small_anchor"]:
    ckpt = CKPT_DIR / f"{run_name}_best.pt"
    if ckpt.is_file():
        print(f"{ckpt.name}: {compute_sha256(ckpt)} ({ckpt.stat().st_size / (1024**2):.2f} MB)")
    else:
        print(f"{ckpt.name}: NOT FOUND")